# 02 - Data Scoring

This notebook calculates the composite scores required for descriptive statistics and predictive modeling.

1. Load the cleaned survey dataset.
2. Validate the 25 survey item responses.
3. Reverse-code negatively worded items.
4. Calculate the six predictor construct scores.
5. Calculate the overall AI Adoption Score.
6. Review and export the scored dataset.


In [ ]:
from pathlib import Path

import pandas as pd

## Define Project Paths


In [ ]:
cwd = Path.cwd()

project_candidates = [cwd, cwd.parent]

PROJECT_ROOT = next(
    (path for path in project_candidates if (path / "requirements.txt").is_file()),
    cwd,
)

DATA_DIR = PROJECT_ROOT / "data"

INPUT_PATH = DATA_DIR / "cleaned_data.csv"
OUTPUT_PATH = DATA_DIR / "scored_data.csv"

print(f"Project directory: {PROJECT_ROOT}")
print(f"Input file: {INPUT_PATH}")
print(f"Output file: {OUTPUT_PATH}")

## Load the Cleaned Dataset


In [ ]:
if not INPUT_PATH.is_file():
    raise FileNotFoundError(
        f"Cleaned dataset not found: {INPUT_PATH}. "
        "Run 01_data_preprocessing.ipynb first."
    )

df = pd.read_csv(INPUT_PATH)

print(f"Loaded {len(df):,} respondent(s).")
print(f"Dataset shape: {df.shape}")

display(df.head())

## Define Survey Item Groups


In [ ]:
construct_items = {
    "ai_familiarity": [
        "ai_familiarity_main_functions",
        "ai_familiarity_tool_availability",
        "ai_familiarity_task_fit",
    ],
    "perceived_usefulness": [
        "perceived_usefulness_concept_understanding",
        "perceived_usefulness_task_efficiency",
        "perceived_usefulness_work_quality",
    ],
    "perceived_ease_of_use": [
        "perceived_ease_learning",
        "perceived_ease_academic_tasks",
        "perceived_ease_useful_responses",
    ],
    "trust": [
        "trust_information_dependability",
        "trust_explanations",
        "trust_accuracy_concern",
    ],
    "accessibility": [
        "accessibility_reliable_access",
        "accessibility_tool_access",
        "accessibility_cost_barrier",
    ],
    "digital_literacy": [
        "digital_literacy_prompting",
        "digital_literacy_verification",
        "digital_literacy_limitations_bias",
    ],
}

adoption_items = [
    "adoption_explain_concepts",
    "adoption_summarize_materials",
    "adoption_brainstorm_ideas",
    "adoption_writing_support",
    "adoption_create_study_materials",
    "adoption_solve_problems",
    "adoption_plan_research",
]

predictor_items = [column for columns in construct_items.values() for column in columns]

all_items = predictor_items + adoption_items

reverse_coded_items = {
    "trust_accuracy_concern": "trust_accuracy_concern_reversed",
    "accessibility_cost_barrier": "accessibility_cost_barrier_reversed",
}

print(f"Predictor items: {len(predictor_items)}")
print(f"AI Adoption items: {len(adoption_items)}")
print(f"Total survey items: {len(all_items)}")

## Validate Survey Columns


In [ ]:
missing_columns = [column for column in all_items if column not in df.columns]

if missing_columns:
    raise ValueError(f"The following survey columns are missing: {missing_columns}")

if len(all_items) != 25:
    raise ValueError("Expected exactly 25 survey items.")

if len(set(all_items)) != 25:
    raise ValueError("Duplicate survey item columns detected.")

print("All 25 survey item columns are present.")

## Validate Response Values


In [ ]:
validation_results = []

for column in predictor_items:
    values = pd.to_numeric(df[column], errors="coerce")

    validation_results.append(
        {
            "item": column,
            "missing_values": values.isna().sum(),
            "invalid_values": (~values.between(1, 5) & values.notna()).sum(),
        }
    )

for column in adoption_items:
    values = pd.to_numeric(df[column], errors="coerce")

    validation_results.append(
        {
            "item": column,
            "missing_values": values.isna().sum(),
            "invalid_values": (~values.between(0, 5) & values.notna()).sum(),
        }
    )

validation_df = pd.DataFrame(validation_results)

display(validation_df)

missing_total = validation_df["missing_values"].sum()
invalid_total = validation_df["invalid_values"].sum()

if missing_total > 0 or invalid_total > 0:
    raise ValueError(
        f"Validation failed: {missing_total} missing value(s) "
        f"and {invalid_total} invalid value(s). "
        "Review the affected survey responses before proceeding."
    )

print("All survey responses passed validation.")

## Reverse-Code Negatively Worded Items


In [ ]:
for original_column, reversed_column in reverse_coded_items.items():
    df[reversed_column] = 6 - df[original_column]

display(
    df[list(reverse_coded_items.keys()) + list(reverse_coded_items.values())].head()
)

print("Reverse-coding completed.")

## Calculate Predictor Construct Scores


In [ ]:
scoring_items = {
    "ai_familiarity_score": construct_items["ai_familiarity"],
    "perceived_usefulness_score": construct_items["perceived_usefulness"],
    "perceived_ease_of_use_score": construct_items["perceived_ease_of_use"],
    "trust_score": [
        "trust_information_dependability",
        "trust_explanations",
        "trust_accuracy_concern_reversed",
    ],
    "accessibility_score": [
        "accessibility_reliable_access",
        "accessibility_tool_access",
        "accessibility_cost_barrier_reversed",
    ],
    "digital_literacy_score": construct_items["digital_literacy"],
}

for score_column, columns in scoring_items.items():
    df[score_column] = df[columns].mean(axis=1)

construct_score_columns = list(scoring_items.keys())

display(df[construct_score_columns].head())

## Calculate AI Adoption Score


In [ ]:
df["ai_adoption_score"] = df[adoption_items].mean(axis=1)

display(df[adoption_items + ["ai_adoption_score"]].head())

print("AI Adoption Score calculated.")

## Review Calculated Scores


In [ ]:
score_columns = construct_score_columns + ["ai_adoption_score"]

score_summary = df[score_columns].agg(["count", "mean", "std", "min", "max"]).T

display(score_summary)

### Validate Calculated Score Ranges


In [ ]:
for column in construct_score_columns:
    if not df[column].between(1, 5).all():
        raise ValueError(f"Unexpected score range detected in {column}.")

if not df["ai_adoption_score"].between(0, 5).all():
    raise ValueError("Unexpected score range detected in ai_adoption_score.")

print("All calculated scores are within their expected ranges.")

## Export Scored Dataset


In [ ]:
df.to_csv(OUTPUT_PATH, index=False)

print(f"Scored dataset exported to: {OUTPUT_PATH}")
print(f"Dataset shape: {df.shape}")
print(f"Respondents scored: {len(df):,}")

## Final Output Inspection


In [ ]:
display(df[construct_score_columns + ["ai_adoption_score"]].head())

print(f"Total columns: {len(df.columns)}")
print(f"Total respondents: {len(df)}")
print("Scoring process completed successfully.")